In [1]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


# Practice 3 — Data and preprocessing (R1)

> **Team code style (delete this cell before submitting).** The team must be able to defend every line.
>
> - Prefer tools seen in class or named in the statement. If you cannot explain a line in one sentence, rewrite it.
> - Plain `for` loops and `if`s, one step per line, clear variable names. No lambdas, classes or long chains.
> - Same results, simpler code. After simplifying, restart, run all and check that the numbers did not change.
> - Interpretation goes in Markdown cells, in English. Check your notebook with `revisor_notebook.py`.
>
> **Shared names, so that the notebooks can be joined:** `dev` (development table), `X`, `y` (cyber target), `z` (storage target), `latency_ns`, `NOMINAL`, `NUMERIC`, `FEATURES`, `REFERENCE_LEVELS`, `make_preprocessor()`, `folds` (list of `(train_idx, val_idx)`), `CLASS_NAMES` (cyber).
> **Internal R1 draft:** Spanish code comments are intentional for team discussion. Translate them into English when integrating the assessed notebooks.

## Setup

In [2]:
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Estas rutas parten de la carpeta notebooks, donde se ejecuta este archivo.
CYBER_DIR = "../datos/cyber"
FOLDS_DIR = "../folds"

# Challenge A — cyber

## Data

In [3]:
# Cargamos los datos y los archivos auxiliares sin modificar los originales.
dev = pd.read_csv(CYBER_DIR + "/cyber_development.csv")
cyber_groups = pd.read_csv(CYBER_DIR + "/cyber_groups.csv")
cyber_folds = pd.read_csv(FOLDS_DIR + "/cyber_folds.csv")

# Las claves del JSON son texto; las convertimos a los códigos enteros de y.
with open(CYBER_DIR + "/cyber_class_codes.json", encoding="utf-8") as file:
    class_codes = json.load(file)

CLASS_NAMES = {}
for code, name in class_codes.items():
    CLASS_NAMES[int(code)] = name

print("Development table:", dev.shape)
print("Class names:", CLASS_NAMES)

Development table: (34928, 13)
Class names: {0: 'backdoor', 1: 'ddos', 2: 'dos', 3: 'injection', 4: 'mitm', 5: 'normal', 6: 'password', 7: 'ransomware', 8: 'scanning', 9: 'xss'}


In [4]:
# Conservamos los 11 predictores de P2 y separamos el objetivo.
NOMINAL = ["proto", "service", "conn_state"]
NUMERIC = ["duration", "src_bytes", "dst_bytes", "missed_bytes",
           "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes"]
FEATURES = NOMINAL + NUMERIC

expected_columns = ["row_id"] + FEATURES + ["attack_code"]
assert list(dev.columns) == expected_columns, "Unexpected cyber schema"

# Seleccionamos las columnas de forma explícita: los metadatos no entran en X.
X = dev[FEATURES].copy()
y = dev["attack_code"].copy()
for column in ["row_id", "feature_group", "config_group", "fold_id", "attack_code"]:
    assert column not in X.columns, "Forbidden predictor: " + column

assert X.shape[1] == 11, "Expected 11 source predictors"
assert not dev.isna().any().any(), "Missing values in development data"
assert np.isfinite(X[NUMERIC].to_numpy()).all(), "Non-finite numeric predictors"
assert (X[NUMERIC] >= 0).all().all(), "Negative values are incompatible with log1p"
assert pd.api.types.is_integer_dtype(y), "Class codes must be integers"
assert sorted(y.unique()) == sorted(CLASS_NAMES), "Class codes do not match JSON"

print("Predictors:", X.shape)
print("Target:", y.name)
print("Schema, predictor and target checks passed.")

Predictors: (34928, 11)
Target: attack_code
Schema, predictor and target checks passed.


In [5]:
# Antes de usar posiciones, comprobamos que los tres archivos están alineados.
assert list(cyber_groups.columns) == ["row_id", "feature_group"], "Unexpected group schema"
assert list(cyber_folds.columns) == ["row_id", "fold_id"], "Unexpected fold schema"
assert dev["row_id"].is_unique, "Duplicate development row_id"
assert cyber_groups["row_id"].is_unique, "Duplicate group row_id"
assert cyber_folds["row_id"].is_unique, "Duplicate fold row_id"
assert not cyber_groups.isna().any().any(), "Missing group metadata"
assert not cyber_folds.isna().any().any(), "Missing fold metadata"
assert dev["row_id"].equals(cyber_groups["row_id"]), "Group rows are not aligned"
assert dev["row_id"].equals(cyber_folds["row_id"]), "Fold rows are not aligned"
assert sorted(cyber_folds["fold_id"].unique()) == [0, 1, 2, 3, 4], "Expected five folds"

print("Row identifiers are unique and aligned across all three tables.")
print("Feature groups:", cyber_groups["feature_group"].nunique())

Row identifiers are unique and aligned across all three tables.
Feature groups: 34883


In [6]:
# Leemos la asignación oficial: no generamos ni cambiamos las particiones.
# np.flatnonzero devuelve las posiciones que cumplen cada condición.
folds = []
fold_rows = []
for fold_id in range(5):
    train_idx = np.flatnonzero(cyber_folds["fold_id"].to_numpy() != fold_id)
    val_idx = np.flatnonzero(cyber_folds["fold_id"].to_numpy() == fold_id)

    # Un grupo nunca puede aparecer a ambos lados de la misma partición.
    train_groups = set(cyber_groups.iloc[train_idx]["feature_group"])
    val_groups = set(cyber_groups.iloc[val_idx]["feature_group"])
    assert train_groups.isdisjoint(val_groups), "Group leakage in fold " + str(fold_id)
    assert len(train_idx) + len(val_idx) == len(dev), "Incomplete partition"

    folds.append((train_idx, val_idx))
    fold_rows.append({"fold_id": fold_id, "training_rows": len(train_idx),
                      "validation_rows": len(val_idx)})

print(pd.DataFrame(fold_rows).to_string(index=False))
print("All five canonical folds passed the group separation checks.")
# Todavía no ajustamos ningún encoder, escalador ni modelo.

 fold_id  training_rows  validation_rows
       0          27942             6986
       1          27942             6986
       2          27943             6985
       3          27942             6986
       4          27943             6985
All five canonical folds passed the group separation checks.


The cyber development table contains 34,928 rows and 11 source predictors (3 nominal and 8 numeric). The target is `attack_code`, with the 10 semantic class names supplied in the JSON file. No missing values were found.

Identifiers, grouping metadata and fold assignments are kept outside `X`. The existing five canonical folds are read from the course CSV, with row alignment and group separation checked before use. No preprocessing or model has been fitted in this block.

## Declared reference levels (statement, Sections 1.1 and 2)

In [7]:
# El orden debe coincidir con NOMINAL: proto, service y conn_state.
# Elegimos los niveles más frecuentes; no se eligen por resultados del modelo.
REFERENCE_LEVELS = ["tcp", "-", "SF"]

reference_rows = []
for column, reference in zip(NOMINAL, REFERENCE_LEVELS):
    row = {"predictor": column, "reference": reference}
    for fold_id, (train_idx, val_idx) in enumerate(folds):
        # El nivel eliminado debe existir en el entrenamiento de cada fold.
        training_count = int((X.iloc[train_idx][column] == reference).sum())
        assert training_count > 0, "Reference missing from training: " + column
        row["training_count_fold_" + str(fold_id)] = training_count
    reference_rows.append(row)

print(pd.DataFrame(reference_rows).to_string(index=False))

 predictor reference  training_count_fold_0  training_count_fold_1  training_count_fold_2  training_count_fold_3  training_count_fold_4
     proto       tcp                  22721                  22737                  22729                  22728                  22777
   service         -                  13152                  13189                  13153                  13146                  13176
conn_state        SF                  11748                  11684                  11725                  11712                  11691


**Reference levels and their reasons.** The declared levels are `proto=tcp`,
`service=-` and `conn_state=SF`, in the order of `NOMINAL`. Each is the most
frequent level of its predictor in the development table, and the counts above
confirm that each is present in every fold's training portion. These are fixed
representation choices, not choices selected by model scores. The `-` service
label is a recorded category, not a missing value. Dropping these indicators
makes each remaining level's coefficient a contrast with its declared reference.

**Unseen-level policy.** With `handle_unknown="ignore"`, an unseen level is
encoded as zeros for that predictor. Because the reference indicator is dropped,
this is also the reference level's encoding: an unseen level is treated as the
declared reference for that predictor. This is a fallback, not evidence that the
levels have the same behaviour. In particular, `service=gssapi` is absent from
training in fold 3, so its validation row is encoded like `service=-`. The
encoder learns its vocabulary only from each fold's training rows.

## Preprocessing block (publish for R2 and R4)

In [8]:
def make_preprocessor():
    # Mismo patrón que en el notebook 02: un bloque por tipo de columna.
    # Conservamos log1p de P2; StandardScaler aprende solo del entrenamiento.
    numeric_steps = Pipeline([
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])

    # Se elimina una categoría por predictor, en el orden de NOMINAL.
    nominal_steps = OneHotEncoder(
        drop=REFERENCE_LEVELS,
        handle_unknown="ignore",
        sparse_output=False,
    )

    preprocessor = ColumnTransformer([
        ("nominal", nominal_steps, NOMINAL),
        ("numeric", numeric_steps, NUMERIC),
    ])
    # Se devuelve sin ajustar para que R2/R4 lo incluyan en su Pipeline.
    return preprocessor

**Scaling decision.** All eight numeric predictors retain the frozen P2
`log1p` representation, followed by `StandardScaler`. The logarithm is defined
at zero and compresses the wide ranges of counts and duration. Standardisation
then expresses each numeric predictor in standard deviations of its
log-transformed training values. A one-unit change therefore means one training
standard deviation on the `log1p` scale, not one unit of the original quantity.
The categorical indicators remain 0/1 and are not standardised.

`make_preprocessor()` returns a new, unfitted transformer. R2/R4 must place it
inside their model `Pipeline`, fitted only on each fold's training rows. The
checks below inspect fold-specific fitted state, following notebooks 02 and 05.
`feature_names_out="one-to-one"` preserves the numeric names through `log1p`,
so the transformed columns can later be matched to model coefficients. Numeric
coefficients may be compared on their common scale; their magnitudes must not
be ranked against categorical coefficients.

In [9]:
# Comprobación del bloque: cada fold empieza con un preprocesador nuevo.
preprocessing_rows = []
for fold_id, (train_idx, val_idx) in enumerate(folds):
    preprocessor = make_preprocessor()
    assert not hasattr(preprocessor, "transformers_"), "Expected an unfitted transformer"
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]
    X_train_transformed = preprocessor.fit_transform(X_train)
    X_valid_transformed = preprocessor.transform(X_valid)

    # Como en el notebook 02, inspeccionamos lo que ha aprendido cada bloque.
    encoder = preprocessor.named_transformers_["nominal"]
    scaler = preprocessor.named_transformers_["numeric"].named_steps["scale"]
    training_numeric = np.log1p(X_train[NUMERIC].to_numpy())
    assert np.allclose(scaler.mean_, training_numeric.mean(axis=0)), "Wrong scaler mean"
    assert np.allclose(scaler.scale_, training_numeric.std(axis=0)), "Wrong scaler scale"
    for column, categories, dropped in zip(NOMINAL, encoder.categories_, encoder.drop_idx_):
        assert list(categories) == sorted(X_train[column].unique()), "Wrong vocabulary"
        position = NOMINAL.index(column)
        assert categories[dropped] == REFERENCE_LEVELS[position], "Wrong dropped level"

    # Validación usa transform, nunca fit: conserva las columnas del entrenamiento.
    feature_names = preprocessor.get_feature_names_out()
    assert X_train_transformed.shape[1] == len(feature_names), "Missing feature names"
    assert X_valid_transformed.shape[1] == len(feature_names), "Inconsistent columns"
    assert np.isfinite(X_train_transformed).all(), "Non-finite training values"
    assert np.isfinite(X_valid_transformed).all(), "Non-finite validation values"
    preprocessing_rows.append({"fold_id": fold_id, "output_features": len(feature_names),
                               "training_state_verified": True})

print(pd.DataFrame(preprocessing_rows).to_string(index=False))
print("Transformed feature names (last fold):", list(feature_names))

 fold_id  output_features  training_state_verified
       0               30                     True
       1               30                     True
       2               30                     True
       3               29                     True
       4               30                     True
Transformed feature names (last fold): ['nominal__proto_icmp', 'nominal__proto_udp', 'nominal__service_dce_rpc', 'nominal__service_dns', 'nominal__service_ftp', 'nominal__service_gssapi', 'nominal__service_http', 'nominal__service_smb', 'nominal__service_smb;gssapi', 'nominal__service_ssl', 'nominal__conn_state_OTH', 'nominal__conn_state_REJ', 'nominal__conn_state_RSTO', 'nominal__conn_state_RSTOS0', 'nominal__conn_state_RSTR', 'nominal__conn_state_RSTRH', 'nominal__conn_state_S0', 'nominal__conn_state_S1', 'nominal__conn_state_S2', 'nominal__conn_state_S3', 'nominal__conn_state_SH', 'nominal__conn_state_SHR', 'numeric__duration', 'numeric__src_bytes', 'numeric__dst_bytes', 'numeric__m

C:\Users\marinagarcia\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [10]:
# Comprobamos el caso real descrito en el enunciado, sin inventar otra categoría.
train_idx, val_idx = folds[3]
preprocessor = make_preprocessor()
preprocessor.fit(X.iloc[train_idx])
encoder = preprocessor.named_transformers_["nominal"]
assert "gssapi" not in encoder.categories_[1], "gssapi should be unseen in fold 3"

validation_rows = X.iloc[val_idx]
unseen_row = validation_rows.loc[validation_rows["service"] == "gssapi", NOMINAL]
assert len(unseen_row) == 1, "Expected one unseen gssapi row"
reference_row = unseen_row.copy()
reference_row["service"] = REFERENCE_LEVELS[1]

# Solo cambiamos service: ambas filas deben tener la misma codificación nominal.
assert np.array_equal(encoder.transform(unseen_row), encoder.transform(reference_row))
print("Fold 3: unseen service=gssapi is encoded like reference service=-.")

Fold 3: unseen service=gssapi is encoded like reference service=-.


C:\Users\marinagarcia\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


The checks pass on all five canonical folds. The transformed matrix has 30 columns in folds 0, 1, 2 and 4, and 29 in fold 3 because its training vocabulary excludes `service=gssapi`. Training and validation have the same columns within each fold. A different vocabulary across folds is expected; coefficients must be read using each fitted preprocessor's feature names.

The unknown-category warning in fold 3 is expected: the encoder applies the declared all-zero fallback. The validation rows are transformed without refitting. These are preprocessing checks, not model performance results.

# Challenge B — storage

## Data

In [11]:
# Este bloque usa los mismos nombres que cyber para facilitar la integración.
# R5 copiará cada reto a su notebook final, junto con el setup compartido.
STORAGE_DIR = "../datos/storage"
dev = pd.read_csv(STORAGE_DIR + "/storage_development.csv")
storage_groups = pd.read_csv(STORAGE_DIR + "/storage_groups.csv")
storage_folds = pd.read_csv(FOLDS_DIR + "/storage_folds.csv")

# raid es una etiqueta nominal, aunque sus niveles contengan números.
NOMINAL = ["load_type", "io_type", "raid", "device_type"]
NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]
FEATURES = NOMINAL + NUMERIC

# El CSV tiene otro orden de columnas: comparamos los nombres ordenados.
expected_columns = ["row_id"] + FEATURES + ["latency_ns"]
assert sorted(dev.columns) == sorted(expected_columns), "Unexpected storage schema"
assert "iops" not in dev.columns, "Excluded column iops must not be present"
X = dev[FEATURES].copy()
latency_ns = dev["latency_ns"].copy()

# El objetivo debe ser finito y positivo antes de calcular el logaritmo.
assert not dev.isna().any().any(), "Missing values in development data"
assert np.isfinite(X[NUMERIC].to_numpy()).all(), "Non-finite numeric predictors"
assert np.isfinite(latency_ns).all(), "Non-finite target values"
assert (latency_ns > 0).all(), "Latency must be strictly positive"
z = np.log10(latency_ns)

for column in ["row_id", "config_group", "feature_group", "fold_id", "latency_ns", "iops"]:
    assert column not in X.columns, "Forbidden predictor: " + column
assert X.shape[1] == 9, "Expected nine source predictors"

print("Development table:", dev.shape)
print("Predictors:", X.shape)
print("Target: z = log10(latency_ns)")
print("Schema, predictor and target checks passed.")

Development table: (12190, 11)
Predictors: (12190, 9)
Target: z = log10(latency_ns)
Schema, predictor and target checks passed.


In [12]:
# Igual que en cyber, verificamos el orden antes de usar índices por posición.
assert list(storage_groups.columns) == ["row_id", "config_group"], "Unexpected group schema"
assert list(storage_folds.columns) == ["row_id", "fold_id"], "Unexpected fold schema"
assert dev["row_id"].is_unique, "Duplicate development row_id"
assert storage_groups["row_id"].is_unique, "Duplicate group row_id"
assert storage_folds["row_id"].is_unique, "Duplicate fold row_id"
assert not storage_groups.isna().any().any(), "Missing group metadata"
assert not storage_folds.isna().any().any(), "Missing fold metadata"
assert dev["row_id"].equals(storage_groups["row_id"]), "Group rows are not aligned"
assert dev["row_id"].equals(storage_folds["row_id"]), "Fold rows are not aligned"
assert sorted(storage_folds["fold_id"].unique()) == [0, 1, 2, 3, 4], "Expected five folds"

print("Row identifiers are unique and aligned across all three tables.")
print("Configuration groups:", storage_groups["config_group"].nunique())

Row identifiers are unique and aligned across all three tables.
Configuration groups: 2438


In [13]:
# Conservamos la asignación canónica; no construimos un GroupKFold nuevo.
folds = []
fold_rows = []
for fold_id in range(5):
    train_idx = np.flatnonzero(storage_folds["fold_id"].to_numpy() != fold_id)
    val_idx = np.flatnonzero(storage_folds["fold_id"].to_numpy() == fold_id)

    # Las réplicas de una configuración deben quedar siempre juntas.
    train_groups = set(storage_groups.iloc[train_idx]["config_group"])
    val_groups = set(storage_groups.iloc[val_idx]["config_group"])
    assert train_groups.isdisjoint(val_groups), "Group leakage in fold " + str(fold_id)
    assert len(train_idx) + len(val_idx) == len(dev), "Incomplete partition"

    folds.append((train_idx, val_idx))
    fold_rows.append({"fold_id": fold_id, "training_rows": len(train_idx),
                      "validation_rows": len(val_idx)})

print(pd.DataFrame(fold_rows).to_string(index=False))
print("All five canonical folds passed the group separation checks.")

 fold_id  training_rows  validation_rows
       0           9750             2440
       1           9750             2440
       2           9750             2440
       3           9755             2435
       4           9755             2435
All five canonical folds passed the group separation checks.


The storage development table contains 12,190 rows and nine source predictors: four nominal and five numeric. There are 2,438 configuration groups. Identifiers, grouping metadata, fold assignments and the target are excluded from `X`; `iops` is absent. The canonical assignments keep configurations separate between training and validation.

The target is `z = log10(latency_ns)`, as fixed in P2. The original `latency_ns` values are retained for physical-scale evaluation. R3 must compute predictions in nanoseconds as `10 ** z_hat` before calculating physical errors. Exponentiating a log-scale error does not produce an error in nanoseconds.

## Declared reference levels (statement, Sections 1.1 and 2)

In [14]:
# El orden coincide con NOMINAL: load_type, io_type, raid y device_type.
# Usamos el nivel más frecuente de cada predictor como referencia fija.
REFERENCE_LEVELS = ["random", "write", "4+1", "nvme"]

reference_rows = []
for column, reference in zip(NOMINAL, REFERENCE_LEVELS):
    row = {"predictor": column, "reference": reference}
    for fold_id, (train_idx, val_idx) in enumerate(folds):
        # Cada referencia debe aparecer en los cinco entrenamientos.
        training_count = int((X.iloc[train_idx][column] == reference).sum())
        assert training_count > 0, "Reference missing from training: " + column
        row["training_count_fold_" + str(fold_id)] = training_count
    reference_rows.append(row)

print(pd.DataFrame(reference_rows).to_string(index=False))

  predictor reference  training_count_fold_0  training_count_fold_1  training_count_fold_2  training_count_fold_3  training_count_fold_4
  load_type    random                   6465                   6465                   6450                   6480                   6480
    io_type     write                   5665                   5665                   5620                   5650                   5720
       raid       4+1                   4310                   4400                   4395                   4345                   4350
device_type      nvme                   8125                   8105                   8085                   8120                   8125


**Reference levels and their reasons.** In the order of `NOMINAL`, the declared
references are `load_type=random`, `io_type=write`, `raid=4+1` and
`device_type=nvme`. Each is the most frequent level of its predictor in the
development table and is present in all five training portions, as checked
above. They are fixed baselines for interpretation, not choices selected by
validation scores. Each remaining indicator contrasts its level with the
reference of the same predictor. In particular, RAID schemes are nominal
labels; they are not ordered numbers.

**Unseen-level policy.** `handle_unknown="ignore"` encodes an unseen level as
zeros for that predictor. With the reference indicator dropped, this treats it
as the declared reference. It is a fallback rule, not an assertion that the
levels are equivalent. The vocabulary is learned only from the training rows
of each fold, just as in cyber.

## Preprocessing block (publish for R3 and R4)

In [15]:
def make_preprocessor():
    # Mismo ColumnTransformer del notebook 02, adaptado a las columnas de storage.
    # Eliminamos una categoría de referencia por predictor nominal, como exige P3.
    nominal_steps = OneHotEncoder(
        drop=REFERENCE_LEVELS,
        handle_unknown="ignore",
        sparse_output=False,
    )

    # En storage se estandarizan los valores originales, sin log1p en X.
    numeric_steps = StandardScaler()

    preprocessor = ColumnTransformer([
        ("nominal", nominal_steps, NOMINAL),
        ("numeric", numeric_steps, NUMERIC),
    ])
    # R3/R4 deben incluir este objeto sin ajustar en el Pipeline de cada modelo.
    return preprocessor

**Scaling decision.** The five numeric predictors use `StandardScaler`, fitted
only on the training rows of each canonical fold. A one-unit increase in a
transformed numeric predictor means one training standard deviation in that
original predictor, with the other inputs held fixed. No `log1p` predictor
transformation is introduced for storage. The target transformation
`z = log10(latency_ns)` is separate from this preprocessor.

The nominal predictors use one-hot encoding with a declared reference removed;
their indicators remain 0/1. No imputation is added because the data checks
confirm there are no missing values. Numeric coefficients share a common
standardised scale and can be compared within that group. Categorical
coefficients describe contrasts with their own references; their magnitudes
must not be ranked against numeric coefficients.

`make_preprocessor()` returns a fresh, unfitted transformer. R3/R4 must place
it inside each model's `Pipeline`, so the encoder vocabulary and scaler state
are learned only from that fold's training portion. Validation uses the same
fitted state without refitting. The unregularised, Ridge and Lasso models must
use this same representation.

In [16]:
# Comprobación del bloque: cada fold empieza con un preprocesador nuevo.
preprocessing_rows = []
for fold_id, (train_idx, val_idx) in enumerate(folds):
    preprocessor = make_preprocessor()
    assert not hasattr(preprocessor, "transformers_"), "Expected an unfitted transformer"
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]
    X_train_transformed = preprocessor.fit_transform(X_train)
    X_valid_transformed = preprocessor.transform(X_valid)

    # Como en el notebook 02, inspeccionamos lo que ha aprendido cada bloque.
    encoder = preprocessor.named_transformers_["nominal"]
    scaler = preprocessor.named_transformers_["numeric"]
    training_numeric = X_train[NUMERIC].to_numpy()
    assert np.allclose(scaler.mean_, training_numeric.mean(axis=0)), "Wrong scaler mean"
    assert np.allclose(scaler.scale_, training_numeric.std(axis=0)), "Wrong scaler scale"
    for column, categories, dropped in zip(NOMINAL, encoder.categories_, encoder.drop_idx_):
        assert list(categories) == sorted(X_train[column].unique()), "Wrong vocabulary"
        position = NOMINAL.index(column)
        assert categories[dropped] == REFERENCE_LEVELS[position], "Wrong dropped level"

    # Validación usa transform, nunca fit: conserva las columnas del entrenamiento.
    feature_names = preprocessor.get_feature_names_out()
    assert X_train_transformed.shape[1] == len(feature_names), "Missing feature names"
    assert X_valid_transformed.shape[1] == len(feature_names), "Inconsistent columns"
    assert np.isfinite(X_train_transformed).all(), "Non-finite training values"
    assert np.isfinite(X_valid_transformed).all(), "Non-finite validation values"
    preprocessing_rows.append({"fold_id": fold_id, "output_features": len(feature_names),
                               "training_state_verified": True})

print(pd.DataFrame(preprocessing_rows).to_string(index=False))
print("Transformed feature names (last fold):", list(feature_names))

 fold_id  output_features  training_state_verified
       0               13                     True
       1               13                     True
       2               13                     True
       3               13                     True
       4               13                     True
Transformed feature names (last fold): ['nominal__load_type_sequential', 'nominal__io_type_read', 'nominal__raid_1+1', 'nominal__raid_2+1', 'nominal__raid_2+2', 'nominal__raid_4+2', 'nominal__raid_8+2', 'nominal__device_type_hdd', 'numeric__block_size', 'numeric__n_jobs', 'numeric__iodepth', 'numeric__read_fraction', 'numeric__n_disks']


In [17]:
# Verificamos la representación del objetivo y su vuelta a la escala física.
assert np.isfinite(z).all(), "Non-finite log target"
assert np.allclose(10 ** z, latency_ns), "Target inversion does not recover latency"
print("Target check passed: 10 ** z recovers latency_ns.")

Target check passed: 10 ** z recovers latency_ns.


The preprocessing checks pass on all five canonical folds. Each fitted scaler matches the means and standard deviations of its own training rows, and each encoder vocabulary matches the categories in those rows. The declared reference indicators are removed. Training and validation matrices are finite and have the same 13 output columns in every fold (8 categorical indicators and 5 numeric predictors).

The target inversion check confirms that `10 ** z` recovers the original `latency_ns`. This verifies the representation, not predictive performance. R3 must still invert its model predictions before computing physical-scale errors.

## `features.md`

The dated Practice 3 entry is recorded in `../plantillas/features.md`. It preserves the P2 entries and documents the declared references, encoding policy, retained numeric transformations and unchanged predictive-information basis.